# Unidad 1: Herramientas para la Optimización
## Libro Interactivo Computacional — IPN

**Texto Base Obligatorio:** David G. Luenberger & Yinyu Ye, *Linear and Nonlinear Programming* (4ta Edición, Springer 2016).

Este notebook integra las notas computacionales interactivas de la **Unidad 1**, cubriendo:
1. **1.1 Revisión de Álgebra Lineal y Análisis** (Notación, Espacio $E^n$, Cauchy-Schwarz, Proyecciones Ortogonales, Formas Cuadráticas, Hessiano, Taylor y Factorización $LU$).
2. **1.2 Conjuntos Convexos** (Variedades lineales, Hiperplanos, Semiespacios, Teoremas 1, 2 y 3 de Separación y Puntos Extremos de Politopos).
3. **1.3 Elementos para la Optimización** (Filosofía de modelado, Escala y Esparcidad, Algoritmos Iterativos y Tasas Canónicas, Forma Estándar de PL y Soluciones Básicas Factibles).

--- 
## 1.1 Álgebra Lineal: Espacios $E^n$, Proyección Ortogonal y Formas Cuadráticas

### Descomposición Ortogonal Única
Dado un subespacio $M \subset E^n$ y su complemento ortogonal $M^\perp$, todo vector $\mathbf{x} \in E^n$ se descompone de manera única como:
$$\mathbf{x} = \mathbf{a} + \mathbf{b}, \quad \text{con } \mathbf{a} \in M, \; \mathbf{b} \in M^\perp$$
donde $\mathbf{a} = \mathbf{P}_M \mathbf{x}$ con $\mathbf{P}_M = \mathbf{V}(\mathbf{V}^T\mathbf{V})^{-1}\mathbf{V}^T$ para cualquier base $\mathbf{V}$ de $M$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Verificación de la Desigualdad de Cauchy-Schwarz: |x^T y| <= ||x|| ||y||
x = np.array([3.0, 1.0, 4.0])
y = np.array([1.0, 2.0, 2.0])

prod_escalar = np.dot(x, y)
norma_x = np.linalg.norm(x)
norma_y = np.linalg.norm(y)

print(f"Vector x: {x}")
print(f"Vector y: {y}")
print(f"Producto escalar x^T y = {prod_escalar:.4f}")
print(f"Cota superior ||x|| * ||y|| = {norma_x * norma_y:.4f}")
print(f"¿Cumple Cauchy-Schwarz?: {abs(prod_escalar) <= norma_x * norma_y}")

In [ ]:
# 2. Proyección Ortogonal sobre subespacio M
V = np.array([[1.0, 0.0],
              [1.0, 1.0],
              [0.0, 1.0]])  # Base de M en E^3

# Matriz de proyección P = V (V^T V)^(-1) V^T
P = V @ np.linalg.inv(V.T @ V) @ V.T
a = P @ x         # Proyección en M
b = x - a         # Componente en M_perp

print(f"Proyección a en M: {np.round(a, 4)}")
print(f"Componente b en M_perp: {np.round(b, 4)}")
print(f"Comprobación ortogonalidad a^T b = {np.dot(a, b):.2e}")
print(f"Reconstrucción a + b = {np.round(a + b, 4)}")

--- 
## 1.1.5 Factorización $LU$ de Gauss (Luenberger Apéndice C)

La resolución de $\mathbf{A}\mathbf{x} = \mathbf{b}$ mediante la descomposición $\mathbf{A} = \mathbf{L}\mathbf{U}$ se realiza en dos etapas:
1. **Sustitución hacia adelante:** $\mathbf{L}\mathbf{y} = \mathbf{b}$
2. **Sustitución hacia atrás:** $\mathbf{U}\mathbf{x} = \mathbf{y}$

In [ ]:
def lu_decomposition(A):
    A_curr = np.array(A, dtype=float)
    n = A_curr.shape[0]
    L = np.eye(n)
    for k in range(n - 1):
        for i in range(k + 1, n):
            factor = A_curr[i, k] / A_curr[k, k]
            L[i, k] = factor
            A_curr[i, k:] -= factor * A_curr[k, k:]
    U = A_curr
    return L, U

A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])
b = np.array([5.0, 13.0, 37.0])

L, U = lu_decomposition(A)
print("Matriz L:")
print(L)
print("\nMatriz U:")
print(U)

# Resolver Ly = b y Ux = y
y = np.linalg.solve(L, b)
x_sol = np.linalg.solve(U, y)
print(f"\nSolución calculada x = {x_sol}")
print(f"Residuo ||Ax - b|| = {np.linalg.norm(A @ x_sol - b):.2e}")

--- 
## 1.2 Geometría Convexa y Teorema 1 de Separación de Luenberger

**Teorema 1 (Luenberger Apéndice B.3):**
Sea $C$ un conjunto convexo y sea $\mathbf{y} \notin \bar{C}$. Si $\delta = \inf_{\mathbf{x} \in C} \|\mathbf{x} - \mathbf{y}\| > 0$ se alcanza en $\mathbf{x}_0 \in \partial C$, entonces definiendo $\mathbf{a} = \mathbf{x}_0 - \mathbf{y}$ se cumple que:
$$\mathbf{a}^T\mathbf{y} < \inf_{\mathbf{x} \in C} \mathbf{a}^T\mathbf{x}$$

In [ ]:
from scipy.optimize import minimize

# Elipsoide: (x1/2)^2 + x2^2 <= 1
def ellipsoid(x):
    return (x[0]/2.0)**2 + x[1]**2 - 1.0

y_ext = np.array([2.5, 1.8])
res = minimize(lambda x: np.sum((x - y_ext)**2), [0, 0], constraints={'type': 'ineq', 'fun': lambda x: -ellipsoid(x)})
x0 = res.x
a = x0 - y_ext
delta = np.linalg.norm(x0 - y_ext)

print(f"Punto exterior y = {y_ext}")
print(f"Punto frontera de mínima distancia x0 = {np.round(x0, 4)}")
print(f"Distancia mínima delta = {delta:.4f}")
print(f"Vector normal a = x0 - y = {np.round(a, 4)}")
print(f"a^T y = {np.dot(a, y_ext):.4f} < a^T x0 = {np.dot(a, x0):.4f}")

--- 
## 1.3 Modelado en Programación Lineal y Soluciones Básicas

Cálculo de todas las bases y **Soluciones Básicas Factibles (SBF)** para conectar el álgebra matricial con los puntos extremos del poliedro.

In [ ]:
from itertools import combinations

# Sistema en forma estándar con 2 restricciones y 4 variables (con holguras):
# x1 + 2 x2 + x3 = 6
# 2 x1 + x2 + x4 = 6
A = np.array([[1.0, 2.0, 1.0, 0.0],
              [2.0, 1.0, 0.0, 1.0]])
b = np.array([6.0, 6.0])

print("Cálculo de todas las Soluciones Básicas:")
for cols in combinations(range(4), 2):
    B = A[:, cols]
    if abs(np.linalg.det(B)) > 1e-6:
        xB = np.linalg.solve(B, b)
        x_full = np.zeros(4)
        x_full[list(cols)] = xB
        is_sbf = np.all(x_full >= 0)
        tag = '--> [SBF / Vértice]' if is_sbf else '[No factible]'
        print(f"Base {cols}: x = {x_full} {tag}")